## Missing Data and Time Series in PySpark

### Missing Data

* Often data sources are incomplete
* There are 3 options for filling in missing data:
      1. Just keep the missing data points.
      2. Drop the missing data points/row.
      3. Fill them in with some other value.

#### Keeping the missing data

A few machine learning algorithms can easily deal with missing data, let's see what it looks like:

In [1]:
!curl https://raw.githubusercontent.com/markumreed/colab_pyspark/main/missing_data.csv >> missing_data.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    63  100    63    0     0    171      0 --:--:-- --:--:-- --:--:--   171


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('missing_data').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 20:48:25 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/14 20:48:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 20:48:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv('missing_data.csv', header=True, inferSchema=True)

In [4]:
df.show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob| NULL|
|id002| NULL| NULL|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



In [5]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sales: double (nullable = true)



#### Drop the missing data

You can use the `.na` functions for missing data. The `drop` command has the following parameters:

`df.na.drop(how='any', thresh=None, subset=None)`

* param how: 'any' or 'all'.

  If 'any', drop a row if it contains any nulls.
  If 'all', drop a row only if all its values are null.

* param thresh: int, default None

  If specified, drop rows that have less than `thresh` non-null values.
  This overwrites the `how` parameter.

* param subset:
  optional list of column names to consider.

In [6]:
df.na.drop().show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id004|Karen|404.0|
+-----+-----+-----+



In [7]:
df.na.drop(thresh=2).show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob| NULL|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



In [8]:
df.na.drop(subset=['sales']).show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



In [9]:
df.na.drop(how='any').show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id004|Karen|404.0|
+-----+-----+-----+



In [10]:
df.na.drop(how='all').show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob| NULL|
|id002| NULL| NULL|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



#### Fill the missing values

We can also fill the missing values with new values. If you have multiple nulls across multiple data types, Spark is actually smart enough to match up the data types. For example:

In [11]:
df.na.fill('SOME VALUE').show()

+-----+----------+-----+
|   id|      name|sales|
+-----+----------+-----+
|id001|       Bob| NULL|
|id002|SOME VALUE| NULL|
|id003|SOME VALUE|585.0|
|id004|     Karen|404.0|
+-----+----------+-----+



In [12]:
df.na.fill(999).show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob|999.0|
|id002| NULL|999.0|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



In [14]:
df.na.fill("Missing Name", subset=['name']).show()

+-----+------------+-----+
|   id|        name|sales|
+-----+------------+-----+
|id001|         Bob| NULL|
|id002|Missing Name| NULL|
|id003|Missing Name|585.0|
|id004|       Karen|404.0|
+-----+------------+-----+



In [15]:
from pyspark.sql.functions import mean

In [16]:
mean_value = df.select(mean(df['sales'])).collect()

In [20]:
mean_sales_value = mean_value[0][0]
mean_sales_value

494.5

In [21]:
df.na.fill(mean_sales_value, ['sales']).show()

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob|494.5|
|id002| NULL|494.5|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



In [22]:
# Don't do this
df.na.fill(df.select(mean(df['sales'])).collect()[0][0], ['sales']).show()  # Not easy to read

+-----+-----+-----+
|   id| name|sales|
+-----+-----+-----+
|id001|  Bob|494.5|
|id002| NULL|494.5|
|id003| NULL|585.0|
|id004|Karen|404.0|
+-----+-----+-----+



### Dates and TimeStamps

You will often find yourself working with Time and Date information.

In [23]:
!curl https://raw.githubusercontent.com/markumreed/colab_pyspark/main/WMT.csv >> WMT.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 89556  100 89556    0     0   218k      0 --:--:-- --:--:-- --:--:--  218k


In [24]:
spark = SparkSession.builder.appName('walmart_dates').getOrCreate()

26/06/14 21:11:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [25]:
df = spark.read.csv('WMT.csv', inferSchema=True, header=True)

In [26]:
df.show()

+----------+---------+---------+---------+---------+---------+--------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|
+----------+---------+---------+---------+---------+---------+--------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|
|2016-01-22|62.439999|63.259998|62.130001|62.689999|55.632324| 9197500|
|2016-01-25|62.779999|    63.82|62.549999|63.450001|56.306763|12823400|
|2016-01-26|63.360001|64.470001|63.259998|     64.0|56.794834| 9441200|
|2016-01-27|64.099998|    65.18|63.889999|63.950001|56.750477|10214300|
|2016-01-28|64.029999|64.510002|    63.43|64.220001| 56.99007|11278300|
|2016-01-29|    64.75|66.529999|64.739998|66.360001|58.889149|16439100|
|2016-02-01|65.910004|    67.93|65.889999|     67.5| 59.90081|14728400|
|2016-02-02|67.300003|67.839996|66.279999|66.860001|59.332867|13585900|
|2016-02-03|67.309998|     67.5|    65.07|66.269997| 58.80928|12

In [27]:
from pyspark.sql.functions import format_number, dayofmonth, hour, dayofyear, month, year, weekofyear, date_format

In [28]:
df.select(dayofmonth(df['Date'])).show()

+----------------+
|dayofmonth(Date)|
+----------------+
|              20|
|              21|
|              22|
|              25|
|              26|
|              27|
|              28|
|              29|
|               1|
|               2|
|               3|
|               4|
|               5|
|               8|
|               9|
|              10|
|              11|
|              12|
|              16|
|              17|
+----------------+
only showing top 20 rows


In [29]:
df.select(hour(df['Date'])).show()

+----------+
|hour(Date)|
+----------+
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
|         0|
+----------+
only showing top 20 rows


In [30]:
df.select(dayofyear(df['Date'])).show()

+---------------+
|dayofyear(Date)|
+---------------+
|             20|
|             21|
|             22|
|             25|
|             26|
|             27|
|             28|
|             29|
|             32|
|             33|
|             34|
|             35|
|             36|
|             39|
|             40|
|             41|
|             42|
|             43|
|             47|
|             48|
+---------------+
only showing top 20 rows


In [31]:
df.select(month(df['Date'])).show()

+-----------+
|month(Date)|
+-----------+
|          1|
|          1|
|          1|
|          1|
|          1|
|          1|
|          1|
|          1|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
|          2|
+-----------+
only showing top 20 rows


##### Find Avg Closing Price per month.

In [34]:
df.withColumn('Month', month(df['Date'])).show()

+----------+---------+---------+---------+---------+---------+--------+-----+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|Month|
+----------+---------+---------+---------+---------+---------+--------+-----+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|    1|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|    1|
|2016-01-22|62.439999|63.259998|62.130001|62.689999|55.632324| 9197500|    1|
|2016-01-25|62.779999|    63.82|62.549999|63.450001|56.306763|12823400|    1|
|2016-01-26|63.360001|64.470001|63.259998|     64.0|56.794834| 9441200|    1|
|2016-01-27|64.099998|    65.18|63.889999|63.950001|56.750477|10214300|    1|
|2016-01-28|64.029999|64.510002|    63.43|64.220001| 56.99007|11278300|    1|
|2016-01-29|    64.75|66.529999|64.739998|66.360001|58.889149|16439100|    1|
|2016-02-01|65.910004|    67.93|65.889999|     67.5| 59.90081|14728400|    2|
|2016-02-02|67.300003|67.839996|66.279999|66.860001|59.332867|13

In [35]:
df2 = df.withColumn('Month', month(df['Date']))

In [36]:
df2.groupBy('Month').mean()[['avg(Month)', 'avg(Close)']].show()

+----------+------------------+
|avg(Month)|        avg(Close)|
+----------+------------------+
|      12.0|106.02932022330101|
|       1.0| 98.94980368627448|
|       6.0| 92.23028014018693|
|       3.0| 87.44880724770643|
|       5.0| 90.54859816822429|
|       9.0|100.69396066336634|
|       4.0| 91.55893247572817|
|       8.0| 96.97705391071433|
|       7.0| 96.65647596190473|
|      10.0|102.74810810810816|
|      11.0|105.59009729126213|
|       2.0|  89.1636457083333|
+----------+------------------+



In [37]:
res = df2.groupBy('Month').mean()[['avg(Month)', 'avg(Close)']]
res = res.withColumnRenamed('avg(Month)', 'Month')
res = res.select('Month', format_number('avg(Close)', 2).alias('Mean Close')).show()

+-----+----------+
|Month|Mean Close|
+-----+----------+
| 12.0|    106.03|
|  1.0|     98.95|
|  6.0|     92.23|
|  3.0|     87.45|
|  5.0|     90.55|
|  9.0|    100.69|
|  4.0|     91.56|
|  8.0|     96.98|
|  7.0|     96.66|
| 10.0|    102.75|
| 11.0|    105.59|
|  2.0|     89.16|
+-----+----------+

